# 🌿 Embedded XGBoost Dosage Predictor (ESP32 Micro-Model)

This notebook is specifically configured to generate a **Micro-Model** suitable for microcontrollers like the ESP32. 
It tightly constrains the XGBoost architecture (very few trees, shallow depth) so the resulting `.c` file is small (under 200KB) while maintaining as much accuracy as possible.

### Sections:
1. **Setup & Dependencies**
2. **Data Loading**
3. **Data Preprocessing**
4. **Optuna Tuning (Embedded Constraints)**
5. **Evaluation & Visualization**
6. **Save Model (JSON & Optimized ESP32 C Code)**

## 1. Setup & Dependencies

In [ ]:
import subprocess
import sys

def install_packages():
    packages = ['pandas', 'numpy', 'xgboost', 'scikit-learn', 'matplotlib', 'seaborn', 'optuna', 'm2cgen']
    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print("✅ All dependencies installed!")

install_packages()

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import optuna
import m2cgen as m2c
import os
import re

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("✅ Libraries imported successfully!")

## 2. Data Loading

In [ ]:
DATASET_PATH = r"C:\Users\hsu.ss_\Desktop\XGBoost training\Agri3D_LeafyGreens_Dataset_CLEANED.csv"

if not os.path.exists(DATASET_PATH):
    print(f"❌ Error: Dataset not found at {DATASET_PATH}")
else:
    df = pd.read_csv(DATASET_PATH)
    print(f"✅ Dataset loaded successfully! Shape: {df.shape}")

## 3. Data Preprocessing

In [ ]:
df['total_dosage_kg_per_ha'] = (
    df['N_dosage_kg_per_ha'].fillna(0) + 
    df['P2O5_dosage_kg_per_ha'].fillna(0) + 
    df['K2O_dosage_kg_per_ha'].fillna(0)
)

LIQUID_DENSITY_KG_PER_L = 1.30
df['total_dosage_ml_per_ha'] = (df['total_dosage_kg_per_ha'] / LIQUID_DENSITY_KG_PER_L) * 1000
target = 'total_dosage_ml_per_ha'

sensor_features = ['N', 'P', 'K', 'temperature', 'humidity', 'pH']
data = df[sensor_features + [target]].dropna()
X = data[sensor_features]
y = data[target]

print(f"🎯 Target variable: {target}")
print(f"📊 Sensor Features ({len(sensor_features)}): {sensor_features}")

## 4. Optuna Tuning (Embedded Constraints)
To ensure the C file fits on an ESP32 (which only has ~4MB of flash), we restrict the model to a maximum of 40 trees and a max depth of 4. This cuts the code size down by 99% compared to default settings.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    # ULTRA-CONSTRAINED Hyperparameter space for Microcontrollers
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 40),     # Very few trees (drastically reduces file size)
        'max_depth': trial.suggest_int('max_depth', 2, 4),             # Shallow trees (fewer nested if-statements)
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.5, log=True), # Higher learning rate compensates for fewer trees
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    
    model = xgb.XGBRegressor(**param, early_stopping_rounds=10)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    return rmse

print("🔍 Tuning Micro-Model (running 30 trials to find best small model)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print("\n🏆 Best Micro-Model Hyperparameters:")
for key, value in study.best_params.items():
    print(f"   {key}: {value}")

best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['random_state'] = 42

final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train, y_train)
print("✅ Final Micro-Model Training Complete!")

## 5. Evaluation & Visualization

In [ ]:
y_pred = final_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📈 Final Micro-Model Performance:")
print(f"   RMSE: {rmse:.2f} ML/ha")
print(f"   R² Score: {r2:.4f}\n")

importance = final_model.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': sensor_features, 'Importance': importance})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
print("🌟 FEATURE IMPORTANCE:")
display(feature_imp_df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].scatter(y_test, y_pred, alpha=0.5, color='blue', label='Predicted Points')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2, linestyle='--', label='Actual (Perfect Fit)')
axes[0].set_xlabel("Actual Dosage (ML/ha)")
axes[0].set_ylabel("Predicted Dosage (ML/ha)")
axes[0].set_title("Micro-Model Scatter: Actual vs Predicted")
axes[0].legend()
axes[0].grid(True)

num_samples = min(50, len(y_test))
x_axis = np.arange(num_samples)

axes[1].scatter(x_axis, y_test.values[:num_samples], color='red', label='Actual Dosage', s=50, zorder=3)
axes[1].scatter(x_axis, y_pred[:num_samples], color='blue', label='Predicted Dosage', s=50, zorder=3)
axes[1].vlines(x_axis, y_test.values[:num_samples], y_pred[:num_samples], color='gray', alpha=0.5)

axes[1].set_xlabel("Test Sample Index")
axes[1].set_ylabel("Dosage (ML/ha)")
axes[1].set_title(f"Micro-Model Sample Comparison (First {num_samples})")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save Model (JSON & Optimized ESP32 C File)

In [ ]:
SAVE_DIR = r"C:\Users\hsu.ss_\Desktop\XGBoost training"

JSON_MODEL_PATH = os.path.join(SAVE_DIR, "xgboost_fertilizer_model.json")
final_model.save_model(JSON_MODEL_PATH)
print(f"✅ JSON Model saved: {JSON_MODEL_PATH}")

print("\n⚙️ Exporting Optimized Micro-Model to C code for ESP32...")
try:
    c_code = m2c.export_to_c(final_model)
    
    # --- ESP32 OPTIMIZATIONS ---
    # 1. Replace 64-bit 'double' with 32-bit 'float' for faster computation and lower RAM usage on ESP32
    c_code = c_code.replace("double", "float")
    
    # 2. Fix 'nan' compilation errors by explicitly handling math.h and NAN macros
    c_code = re.sub(r'\bnan\b', 'NAN', c_code)
    
    esp32_header = """
// Auto-generated Optimized XGBoost Micro-Model C code for ESP32
// Input features: """ + ", ".join(sensor_features) + """

#include <math.h>
#include <stdint.h>

#ifndef NAN
    #define NAN (0.0f/0.0f)
#endif
"""
    
    C_MODEL_PATH = os.path.join(SAVE_DIR, "xgboost_fertilizer_model.c")
    with open(C_MODEL_PATH, "w") as f:
        f.write(esp32_header)
        f.write(c_code)
        
    file_size_kb = os.path.getsize(C_MODEL_PATH) / 1024
    print(f"🎉 SUCCESS! C Model generated at: {C_MODEL_PATH}")
    print(f"📦 File Size: {file_size_kb:.2f} KB (Highly Optimized for ESP32 Compilation!)")
    
except Exception as e:
    print(f"❌ Failed to generate C code. Error: {e}")